# Experiment 4: Crossover Operation
**Aim:** To implement the crossover operation for generating new offspring from selected individuals using Python.

**Theory:** Crossover combines genetic material of two parents to produce offspring, exploiting good building blocks. Implemented: single-point, two-point, uniform (binary), arithmetic/blend (real-valued) and Order Crossover (permutation).

In [1]:
import random
random.seed(3)

p1 = [1, 1, 1, 1, 1, 1, 1, 1]
p2 = [0, 0, 0, 0, 0, 0, 0, 0]
print("Parent 1:", p1)
print("Parent 2:", p2)

Parent 1: [1, 1, 1, 1, 1, 1, 1, 1]
Parent 2: [0, 0, 0, 0, 0, 0, 0, 0]


## 1. Single-point crossover

In [2]:
def single_point(a, b):
    pt = random.randint(1, len(a) - 1)
    return a[:pt] + b[pt:], b[:pt] + a[pt:], pt

c1, c2, pt = single_point(p1, p2)
print("cut point:", pt); print("Child 1:", c1); print("Child 2:", c2)

cut point: 2
Child 1: [1, 1, 0, 0, 0, 0, 0, 0]
Child 2: [0, 0, 1, 1, 1, 1, 1, 1]


## 2. Two-point crossover

In [3]:
def two_point(a, b):
    i, j = sorted(random.sample(range(1, len(a)), 2))
    return a[:i] + b[i:j] + a[j:], b[:i] + a[i:j] + b[j:], (i, j)

c1, c2, pts = two_point(p1, p2)
print("cut points:", pts); print("Child 1:", c1); print("Child 2:", c2)

cut points: (5, 7)
Child 1: [1, 1, 1, 1, 1, 0, 0, 1]
Child 2: [0, 0, 0, 0, 0, 1, 1, 0]


## 3. Uniform crossover

In [4]:
def uniform(a, b, p=0.5):
    c1, c2 = [], []
    for x, y in zip(a, b):
        if random.random() < p:
            c1.append(y); c2.append(x)
        else:
            c1.append(x); c2.append(y)
    return c1, c2

c1, c2 = uniform(p1, p2)
print("Child 1:", c1); print("Child 2:", c2)

Child 1: [0, 1, 0, 1, 1, 1, 0, 1]
Child 2: [1, 0, 1, 0, 0, 0, 1, 0]


## 4. Arithmetic crossover (real-valued)
Child₁ = α·P₁ + (1−α)·P₂, Child₂ = (1−α)·P₁ + α·P₂

In [5]:
def arithmetic(a, b, alpha=None):
    alpha = random.random() if alpha is None else alpha
    c1 = [alpha * x + (1 - alpha) * y for x, y in zip(a, b)]
    c2 = [(1 - alpha) * x + alpha * y for x, y in zip(a, b)]
    return c1, c2, alpha

rp1, rp2 = [1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]
c1, c2, a = arithmetic(rp1, rp2)
print("alpha =", round(a, 3)); print("Child 1:", [round(x, 3) for x in c1]); print("Child 2:", [round(x, 3) for x in c2])

alpha = 0.192
Child 1: [4.233, 5.233, 6.233, 7.233]
Child 2: [1.767, 2.767, 3.767, 4.767]


## 5. Order crossover (OX) for permutation encoding

In [6]:
def order_crossover(a, b):
    n = len(a)
    i, j = sorted(random.sample(range(n), 2))
    def make_child(x, y):
        child = [None] * n
        child[i:j + 1] = x[i:j + 1]
        fill = [g for g in y[j + 1:] + y[:j + 1] if g not in child]
        pos = [k for k in list(range(j + 1, n)) + list(range(0, j + 1)) if child[k] is None]
        for k, g in zip(pos, fill):
            child[k] = g
        return child
    return make_child(a, b), make_child(b, a), (i, j)

pa, pb = [1, 2, 3, 4, 5, 6, 7, 8], [3, 7, 5, 1, 6, 8, 2, 4]
c1, c2, seg = order_crossover(pa, pb)
print("segment:", seg); print("Parent A:", pa); print("Parent B:", pb)
print("Child 1:", c1, sorted(c1) == sorted(pa)); print("Child 2:", c2, sorted(c2) == sorted(pa))

segment: (4, 7)
Parent A: [1, 2, 3, 4, 5, 6, 7, 8]
Parent B: [3, 7, 5, 1, 6, 8, 2, 4]
Child 1: [3, 1, 2, 4, 5, 6, 7, 8] True
Child 2: [1, 3, 5, 7, 6, 8, 2, 4] True


## 6. Applying crossover to a mating pool with crossover probability Pc

In [7]:
def apply_crossover(pool, pc=0.8, op=single_point):
    offspring = []
    for i in range(0, len(pool) - 1, 2):
        a, b = pool[i], pool[i + 1]
        if random.random() < pc:
            c1, c2 = op(a, b)[:2]
        else:
            c1, c2 = a[:], b[:]
        offspring += [c1, c2]
    return offspring

pool = [[random.randint(0, 1) for _ in range(8)] for _ in range(6)]
print("Parents:");   [print(" ", p) for p in pool]
print("Offspring:"); [print(" ", c) for c in apply_crossover(pool, pc=0.9)]

Parents:
  [1, 1, 0, 0, 0, 1, 0, 0]
  [0, 0, 1, 0, 1, 1, 1, 1]
  [1, 1, 0, 1, 0, 0, 0, 1]
  [0, 1, 1, 1, 1, 1, 1, 1]
  [0, 1, 0, 1, 0, 1, 0, 0]
  [1, 1, 0, 0, 1, 1, 0, 1]
Offspring:
  [1, 1, 0, 0, 1, 1, 1, 1]
  [0, 0, 1, 0, 0, 1, 0, 0]
  [1, 1, 1, 1, 1, 1, 1, 1]
  [0, 1, 0, 1, 0, 0, 0, 1]
  [0, 1, 0, 1, 0, 1, 0, 1]
  [1, 1, 0, 0, 1, 1, 0, 0]


[None, None, None, None, None, None]

## Conclusion
Single-point, two-point, uniform, arithmetic and order crossover were implemented. Binary/real operators suit bit-string and continuous problems, while OX keeps permutations valid.